# 03. 분류 모델 (지도 학습)

**목적**: 02_preprocessing에서 할당한 그룹 레이블을 타겟으로  
건강 수치가 입력되면 어떤 그룹인지 예측하는 분류 모델 학습

**입력**: health_grouped.csv  
**타겟**: group (정상군/고혈압군/혈당이상군/비만군/복합군 등 8개)  
**출력**: diet_plan_classifier.pkl

## 0. 라이브러리 로드

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import joblib
import os
import warnings

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

warnings.filterwarnings('ignore')

## 1. Google Drive 마운트 및 데이터 로드

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Google Drive 내 데이터 경로 (실제 경로로 수정)
DATA_PATH = '/content/drive/MyDrive/AH_03_06/data/processed/health_grouped.csv'

df = pd.read_csv(DATA_PATH, encoding='utf-8-sig')

print(f'데이터 shape: {df.shape}')
print('\n=== 그룹별 인원 ===')
print(df['group'].value_counts())

## 2. 피처 및 타겟 설정

In [ ]:
FEATURE_COLS = ['sbp_c', 'dbp_c', 'fbs_c', 'bmi_c', 'waist_c']
TARGET_COL   = 'group'

X = df[FEATURE_COLS].copy()
y = df[TARGET_COL].copy()

# 레이블 인코딩
le = LabelEncoder()
y_encoded = le.fit_transform(y)

print('클래스 목록:')
for i, cls in enumerate(le.classes_):
    print(f'  {i}: {cls}')

print(f'\nX shape: {X.shape}')

## 3. 학습/검증 데이터 분리

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded,
    test_size=0.2,
    random_state=42,
    stratify=y_encoded
)

print(f'학습 데이터: {len(X_train):,}명')
print(f'검증 데이터: {len(X_test):,}명')

## 4. 모델 학습 및 비교

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = {}

print('=== 5-Fold Cross Validation ===')
for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='f1_weighted', n_jobs=-1)
    results[name] = scores
    print(f'{name:25s}: mean={scores.mean():.4f}  std={scores.std():.4f}')

In [ ]:
# 모델 비교 시각화
fig, ax = plt.subplots(figsize=(10, 5))
ax.boxplot(results.values(), labels=results.keys(), patch_artist=True,
           boxprops=dict(facecolor='#3498DB', alpha=0.6))
ax.set_title('모델별 F1 Score (5-Fold CV)', fontsize=13, fontweight='bold')
ax.set_ylabel('F1 Score (weighted)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

best_model_name = max(results, key=lambda k: results[k].mean())
print(f'최적 모델: {best_model_name}')

## 5. 최적 모델 최종 평가

In [ ]:
best_model = models[best_model_name]
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

print(f'=== {best_model_name} 평가 결과 ===')
print(f'Accuracy : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score : {f1_score(y_test, y_pred, average="weighted"):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=le.classes_))

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=le.classes_, yticklabels=le.classes_, ax=ax)
ax.set_title('Confusion Matrix', fontsize=13, fontweight='bold')
ax.set_xlabel('예측값')
ax.set_ylabel('실제값')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

## 6. 피처 중요도

In [ ]:
if hasattr(best_model, 'feature_importances_'):
    importances = pd.Series(best_model.feature_importances_, index=FEATURE_COLS)
    importances = importances.sort_values(ascending=True)

    fig, ax = plt.subplots(figsize=(8, 4))
    importances.plot(kind='barh', ax=ax, color='#2ECC71', edgecolor='white')
    ax.set_title('피처 중요도', fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance')
    plt.tight_layout()
    plt.show()
    print(importances.sort_values(ascending=False).round(4))

## 7. 모델 저장

In [ ]:
MODEL_DIR = '/content/drive/MyDrive/AH_03_06/models/diet'
os.makedirs(MODEL_DIR, exist_ok=True)

joblib.dump(best_model, f'{MODEL_DIR}/diet_plan_classifier.pkl')
joblib.dump(le,         f'{MODEL_DIR}/label_encoder.pkl')

print('저장 완료:')
print(f'  {MODEL_DIR}/diet_plan_classifier.pkl')
print(f'  {MODEL_DIR}/label_encoder.pkl')

## 8. 예측 함수 테스트

In [ ]:
def predict_group(sbp, dbp, fbs, bmi, waist, gender=1):
    """
    건강 수치를 입력받아 그룹을 예측합니다.
    sbp, dbp, fbs, bmi, waist: 원본 수치
    """
    # 정상/주의/위험 변환
    sbp_c   = 0 if sbp < 120 else (1 if sbp < 140 else 2)
    dbp_c   = 0 if dbp < 80  else (1 if dbp < 90  else 2)
    fbs_c   = 0 if fbs < 100 else (1 if fbs < 126 else 2)
    bmi_c   = 0 if bmi < 23  else (1 if bmi < 25  else 2)
    waist_c = (2 if waist >= 85 else 0) if gender == 2 else (2 if waist >= 90 else 0)

    input_data = np.array([[sbp_c, dbp_c, fbs_c, bmi_c, waist_c]])
    pred = best_model.predict(input_data)
    proba = best_model.predict_proba(input_data)[0]
    pred_label = le.inverse_transform(pred)[0]

    print(f'변환값: sbp={sbp_c} dbp={dbp_c} fbs={fbs_c} bmi={bmi_c} waist={waist_c}')
    print(f'예측 그룹: {pred_label}')
    print('클래스별 확률:')
    for cls, p in zip(le.classes_, proba):
        if p > 0.01:
            print(f'  {cls}: {p*100:.1f}%')
    return pred_label

print('=== 테스트 1: 정상 수치 ===')
predict_group(sbp=115, dbp=72, fbs=90, bmi=22.0, waist=80)

print('\n=== 테스트 2: 고혈압+비만 ===')
predict_group(sbp=145, dbp=92, fbs=95, bmi=27.0, waist=95)

print('\n=== 테스트 3: 혈당이상 ===')
predict_group(sbp=118, dbp=75, fbs=115, bmi=22.0, waist=80)

## 9. 결과 요약

In [ ]:
print('=' * 60)
print('분류 모델 결과 요약')
print('=' * 60)
print(f'최적 모델  : {best_model_name}')
print(f'Accuracy   : {accuracy_score(y_test, y_pred):.4f}')
print(f'F1 Score   : {f1_score(y_test, y_pred, average="weighted"):.4f}')
print(f'피처 변수  : {FEATURE_COLS}')
print(f'분류 그룹  : {list(le.classes_)}')
print()
print('→ 다음 단계: 04_nutrient_standard.ipynb')
print('=' * 60)